# 9.2 Statistical Learning Theory

**Goal:** state risk and generalization bounds with their quantifiers, loss bounds, and sampling assumptions.

**Prerequisites:** Modules 0–4 and earlier notebooks in this module; advanced notebooks also build on earlier modules 5–8. Read the section assumptions before using a method.

**Runtime:** Python 3.10+ with NumPy, SciPy, and Matplotlib. Install in your chosen environment with `python3 -m pip install numpy scipy matplotlib`, then open in JupyterLab or VS Code with a Python kernel. Use **Restart Kernel and Run All**. Each notebook is self-contained, includes saved outputs, uses synthetic data and a fixed local random seed, and needs no downloads. Numerical simulations illustrate results; they do not prove them or validate real-world assumptions.

## Module navigation

1. [Information Theory](1_Information-Theory.ipynb)
2. [Statistical Learning Theory](2_Statistical-Learning-Theory.ipynb)
3. [Advanced Probability](3_Advanced-Probability.ipynb)
4. [Time-Series Statistics](4_Time-Series-Statistics.ipynb)
5. [Statistical Computing](5_Statistical-Computing.ipynb)
6. [Specialized Statistical Methods](6_Specialized-Statistical-Methods.ipynb)

## Learning route

Read each definition and assumption, work through its calculation, run the code, then attempt the practice before viewing the solutions. Distinguish descriptive association, causal inference, and predictive evaluation throughout.

In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats, optimize, special, linalg
rng = np.random.default_rng(2026)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.titlesize": 12, "font.size": 10})

## 1. Empirical Risk Minimization and Expected Risk

For a hypothesis h and loss ℓ, expected risk is $R(h)=E[\ell(h(X),Y)]$ under the target distribution; empirical risk is $\hat R_n(h)=n^{-1}\sum_i\ell(h(X_i),Y_i)$. ERM selects a hypothesis minimizing training risk over a specified class. Training performance is optimistic after adaptive selection, even if each fixed hypothesis has an unbiased empirical risk under IID data.

**Generalization error** sometimes means expected test risk and sometimes the gap R(h)−R̂(h); state the convention. A train-test gap can reflect complexity, shift, leakage, or finite sampling. Bounds are conditional on a sampling setup and loss/class restrictions, not general deployment guarantees.

## 2. Concentration Inequalities: Hoeffding's Inequality

For independent bounded variables in [0,1], Hoeffding gives $P(|\bar X-E\bar X|\ge\epsilon)\le2e^{-2n\epsilon²}$. For a **fixed hypothesis** with bounded [0,1] loss on IID observations, this bounds its empirical-risk deviation. Unbounded squared loss cannot be inserted into this formula without additional assumptions or truncation.

**Worked example:** n=1000, ε=.05 yields an upper bound 2e⁻⁵≈.01348. The result is an upper bound, not an exact tail probability. It does not apply to a best-of-many selected hypothesis without accounting for selection.

In [2]:
n=1000;eps=.05
print("Fixed bounded-loss Hoeffding bound:",min(1.,2*np.exp(-2*n*eps**2)))

Fixed bounded-loss Hoeffding bound: 0.013475893998170922


## 3. Uniform Convergence

For a finite hypothesis class of size M, a union bound yields
$$P(\sup_h|R(h)-\hat R(h)|>\epsilon)\le2M e^{-2n\epsilon²}.$$
Thus with probability at least 1−δ, every h has error at most $\epsilon=\sqrt{\log(2M/δ)/(2n)}$. An ERM's risk is at most the class-best risk plus 2ε on that event. This is a simultaneous statement that supports selection within the specified class; adding hypotheses after seeing the data can invalidate a naive M count.

Bounds can be loose or vacuous; clipping an upper probability at 1 does not make the underlying guarantee informative. Structure-dependent bounds can improve on simple finite-class counting.

In [3]:
M=100;delta=.05
uniform_eps=np.sqrt(np.log(2*M/delta)/(2*n))
print("Finite-class uniform risk radius:",uniform_eps,"ERM excess bound:",2*uniform_eps)

Finite-class uniform risk radius: 0.06439739761862287 ERM excess bound: 0.12879479523724574


## 4. PAC Learning and VC Dimension

**Probably approximately correct (PAC)** learning asks whether a procedure achieves risk within a specified accuracy with probability at least 1−δ using enough samples under a stated class/distribution setting. Realizable PAC assumes a zero-risk target in the class; agnostic PAC compares with the best achievable class risk. Their sample requirements differ.

**VC dimension** is the largest number of points a binary class can shatter (realize every labeling). Affine halfspaces in Rᵈ have VC dimension d+1; linear algebraic parameter count is not a universal VC formula for arbitrary neural or constrained classes. VC-based results extend uniform-convergence reasoning to infinite classes using growth complexity, under IID and bounded classification-loss conditions.

A finite-class agnostic Hoeffding bound requires n≥log(2M/δ)/(2ε²) for a uniform deviation ε. Guaranteeing ERM excess risk ≤η via this route uses ε=η/2. Do not mix those two accuracy quantities.

In [4]:
eta=.1;eps=eta/2
required=math.ceil(np.log(2*M/delta)/(2*eps**2))
print("Sufficient finite-class n for ERM excess <= .1 via union-Hoeffding:",required)

Sufficient finite-class n for ERM excess <= .1 via union-Hoeffding: 1659


## 5. Chernoff Bounds

Chernoff methods bound tails using exponential moments and optimize the exponent. For a sum S of independent Bernoulli variables with mean μ and δ>0,
$$P(S\ge(1+δ)\mu)\le\left[\frac{e^δ}{(1+δ)^{1+δ}}\right]^\mu.$$
This multiplicative form uses the mean and independence; it is not valid for arbitrary dependent counts merely because they are nonnegative. Hoeffding is additive and bounded-variable based; neither is automatically tighter in every regime.

**Worked example:** S∼Binomial(100,.1), μ=10, δ=1 bounds P(S≥20). Compare with the exact binomial tail and keep event endpoints consistent.

In [5]:
mu=10.;d=1.
chernoff=np.exp(mu*(d-(1+d)*np.log1p(d)))
exact=stats.binom.sf(19,100,.1)
print("Chernoff upper bound:",chernoff,"exact P(S>=20):",exact)
assert exact<=chernoff+1e-12

Chernoff upper bound: 0.02100607470970795 exact P(S>=20): 0.0019785608657712354


## AI/ML application and interpretation

Learning theory clarifies why held-out data and complexity control matter, but does not certify an entire production system under arbitrary shift. State the loss range, hypothesis class, independence, target distribution, and whether the bound covers data-dependent selection before using it to justify ML validation.

## Practice

1. Why does a fixed-hypothesis concentration bound not automatically justify unlimited model search?
2. Which assumptions permit the displayed Hoeffding bound?
3. Distinguish realizable from agnostic PAC.
4. Does a valid bound have to be numerically useful?

## Solutions

1. Selection can exploit deviations among many candidates; simultaneous complexity control is needed.
2. Independent bounded variables, with the stated range; IID is used for interpreting a common target risk.
3. Realizable assumes a perfect class member; agnostic compares with the best member despite noise/misspecification.
4. No. A bound may be very conservative or exceed the maximum meaningful loss/probability.

**Readiness check:** identify the estimand, independent unit, assumptions, uncertainty source, and an important failure mode before using these methods.

**Next:** [Continue studying](3_Advanced-Probability.ipynb).